<a href="https://colab.research.google.com/github/mathivathani-1411/mathivathani-codebooster-2026/blob/main/Day5/Day_5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [31]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error,mean_absolute_error,r2_score
print('All libraries imported succssfully')

All libraries imported succssfully


In [9]:
df= pd.read_csv('student_performance.csv')
print(f'Dataset loaded: {df.shape[0]} students,{df.shape[1]}columns')
print(f'Coulumns: {df.columns.tolist()}')
print(f'\n Missing values:{df.isnull().sum().sum()}')
print('\nFirst rows: ')
df.head()

Dataset loaded: 31 students,13columns
Coulumns: ['student_id', 'name', 'age', 'gender', 'department', 'semester', 'math_score', 'science_score', 'english_score', 'programming_score', 'attendance_percentage', 'city', 'admission_year']

 Missing values:1

First rows: 


,student_id,name,age,gender,department,semester,math_score,science_score,english_score,programming_score,attendance_percentage,city,admission_year
0,1001,Aarav Sharma,19,Male,Computer Science,2,85,78,72,91,92,Mumbai,2023
1,1002,Priya Patel,20,Female,Computer Science,2,76,82,88,79,87,Ahmedabad,2023
2,1003,Rohit Verma,19,Male,Electronics,2,65,74,61,55,78,Delhi,2023
3,1004,Sneha Reddy,20,NaN,Mechanical,2,70,80,75,48,95,Hyderabad,2023
4,1005,Arjun Nair,19,Male,Computer Science,2,92,88,81,95,90,Kochi,2023


In [11]:
print('--- Target column statistics---')
print(df['programming_score'].describe())

--- Target column statistics---
count    31.000000
mean     68.451613
std      21.223947
min      38.000000
25%      49.500000
50%      69.000000
75%      90.000000
max      97.000000
Name: programming_score, dtype: float64


In [16]:
df_ml = df.copy()

# Gender Encoding
le_gender = LabelEncoder()
df_ml['gender_encoded'] = le_gender.fit_transform(df_ml['gender'])

print(f"Gender encoding: {dict(zip(le_gender.classes_, le_gender.transform(le_gender.classes_)))}")

# Department Encoding
le_dept = LabelEncoder()
df_ml['department_encoded'] = le_dept.fit_transform(df_ml['department'])

print(f"Department encoding: {dict(zip(le_dept.classes_, le_dept.transform(le_dept.classes_)))}")

print("\nNew Columns added: gender_encoded, department_encoded")

df_ml[['gender', 'gender_encoded',
       'department', 'department_encoded']].head(5)

Gender encoding: {'Female': np.int64(0), 'Male': np.int64(1), nan: np.int64(2)}
Department encoding: {'Civil': np.int64(0), 'Computer Science': np.int64(1), 'Electronics': np.int64(2), 'Mechanical': np.int64(3)}

New Columns added: gender_encoded, department_encoded


,gender,gender_encoded,department,department_encoded
0,Male,1,Computer Science,1
1,Female,0,Computer Science,1
2,Male,1,Electronics,2
3,NaN,2,Mechanical,3
4,Male,1,Computer Science,1


In [18]:
feature_cols = [
    'math_score',
    'science_score',
    'english_score',
    'attendance_percentage',
    'gender_encoded',
    'department_encoded'
]

X = df_ml[feature_cols]
y = df_ml['programming_score']

print(f'Feature matrix X shape: {X.shape} (students x features)')
print(f'Target vector y shape: {y.shape} (one score per student)')

print(f'\nFeature columns: {feature_cols}')
print('Target column: programming_score')

print(f'\nTarget range: {y.min()} to {y.max()} (mean: {y.mean():.1f})')

Feature matrix X shape: (31, 6) (students x features)
Target vector y shape: (31,) (one score per student)

Feature columns: ['math_score', 'science_score', 'english_score', 'attendance_percentage', 'gender_encoded', 'department_encoded']
Target column: programming_score

Target range: 38 to 97 (mean: 68.5)


In [22]:
X_train,X_test,y_train,y_test=train_test_split(
    X,y,
    test_size=0.2,
    random_state=42
)

print(f'Total students:{len(X)}')
print(f'Training students:{len(X_train)}({len(X_train)/len(X)*100:.0f}%)')
print(f'Testing students:{len(X_test)}({len(X_test)/len(X)*100:.0f}%)')
print(f'\nTraining target range:{y_train.min()}-{y_train.max()}')
print(f'Testing target range:{y_test.min()}-{y_test.max()}')

Total students:31
Training students:24(77%)
Testing students:7(23%)

Training target range:38-96
Testing target range:39-97


In [23]:
scaler = StandardScaler()

# Scale training and testing data
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Features scaled successfully!")

print(f"Training feature mean (should be ≈ 0): {X_train_scaled.mean():.4f}")
print(f"Training feature std (should be ≈ 1): {X_train_scaled.std():.4f}")

Features scaled successfully!
Training feature mean (should be ≈ 0): 0.0000
Training feature std (should be ≈ 1): 1.0000


In [32]:
# STEP 8: TRAIN MODEL 1 : Linear regression

lr_model = LinearRegression()
lr_model.fit(X_train_scaled, y_train)
lr_pred = lr_model.predict(X_test_scaled)

# Evaluate
lr_mae = mean_absolute_error(y_test, lr_pred)
lr_rmse = np.sqrt(mean_squared_error(y_test, lr_pred))
lr_r2 = r2_score(y_test, lr_pred)

print('=== Model 1: Linear Regression ===')
print(f'MAE: {lr_mae:.2f} (prediction are off by {lr_mae:.1f} points on average)')
print(f'RMSE: {lr_rmse:.2f} (error with penalty for large mistakes)')
print(f'R^2: {lr_r2:.4f} ({lr_r2*100:.1f}% of programming score variation explained)')
print()
print('Learned coefficients (importance of each feature):')
for feat, coef in zip(feature_cols, lr_model.coef_):
  print(f'{feat}: {coef:+.3f}')
print(f'Bias (intercept): {lr_model.intercept_:.3f}')

=== Model 1: Linear Regression ===
MAE: 8.04 (prediction are off by 8.0 points on average)
RMSE: 10.09 (error with penalty for large mistakes)
R^2: 0.8055 (80.6% of programming score variation explained)

Learned coefficients (importance of each feature):
math_score: +19.677
science_score: +7.729
english_score: +4.394
attendance_percentage: -13.752
gender_encoded: +1.239
department_encoded: -0.541
Bias (intercept): 68.042


In [33]:
dt_model=DecisionTreeRegressor(
    max_depth=5,
    random_state=42
)

dt_model.fit(X_train_scaled,y_train)
dt_pred=dt_model.predict(X_test_scaled)

dt_mae=mean_absolute_error(y_test,dt_pred)
dt_rmse=np.sqrt(mean_squared_error(y_test,dt_pred))
dt_r2=r2_score(y_test,dt_pred)

print('=== Model 2: Decision Tree (max_depths)===')
print(f'MAE:{dt_mae:.2f}')
print(f'RMSE:{dt_rmse:.2f}')
print(f'R^2:{dt_r2:.4f}')

=== Model 2: Decision Tree (max_depths)===
MAE:9.21
RMSE:14.89
R^2:0.5767
